In [ ]:
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import category_encoders as ce


In [ ]:
batsman = pd.read_csv("/content/batsman_match_final_stage2.csv")
batsman = batsman.sort_values("date")

print("STAGE 1 ✅ Dataset loaded")
print("Date range:", batsman["date"].min(), "→", batsman["date"].max())


STAGE 1 ✅ Dataset loaded
Date range: 2008-04-18 → 2021-04-23


In [ ]:
batsman.columns

Index(['matchid', 'date', 'season', 'venue', 'city', 'batting_team',
       'bowling_team', 'batsman', 'runs', 'balls_faced', 'fours', 'sixes',
       'strike_rate', 'avg_runs_last_5', 'avg_runs_last_10',
       'avg_runs_at_venue', 'matches_at_venue', 'matches_played',
       'career_avg_runs', 'career_avg_balls', 'boundary_rate',
       'recent_form_indicator', 'experience_level'],
      dtype='object')

In [ ]:
!pip install category_encoders


In [ ]:
TARGET = "runs"

DROP_COLS = [
    "runs",
    "batsman",
    "matchid",
    "date",
    "balls_faced",
    "fours",
    "sixes",
    "strike_rate",
    "boundary_rate"
]

X = batsman.drop(columns=DROP_COLS)
y = batsman[TARGET]

print("STAGE 2 ✅ Leakage columns removed")
print("Remaining features:", X.columns.tolist())


STAGE 2 ✅ Leakage columns removed
Remaining features: ['season', 'venue', 'city', 'batting_team', 'bowling_team', 'avg_runs_last_5', 'avg_runs_last_10', 'avg_runs_at_venue', 'matches_at_venue', 'matches_played', 'career_avg_runs', 'career_avg_balls', 'recent_form_indicator', 'experience_level']


In [ ]:
batsman['date'] = pd.to_datetime(batsman['date'])
split_date = '2018-01-01'
X_train = X[batsman["date"] <= split_date]
X_test  = X[batsman["date"] > split_date]

y_train = y[batsman["date"] <= split_date]
y_test  = y[batsman["date"] > split_date]

In [ ]:
categorical_cols = [
    "venue",
    "city",
    "season",
    "batting_team",
    "bowling_team",
    "experience_level"
]

numerical_cols = [

    "avg_runs_last_5",
    "avg_runs_last_10",
    "avg_runs_at_venue",
    "matches_at_venue",
    "matches_played",
    "career_avg_runs",
    "career_avg_balls",
    "recent_form_indicator"
]


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ce.TargetEncoder(), categorical_cols),
        ("num", StandardScaler(), numerical_cols)
    ]
)

feature_pipeline = Pipeline(
    steps=[("preprocessing", preprocessor)]
)

feature_pipeline.fit(X_train, y_train)

joblib.dump(feature_pipeline, "batsman_feature_pipeline.pkl")

print(" Feature pipeline rebuilt & saved")


 Feature pipeline rebuilt & saved


In [ ]:
X_train['season'] = X_train['season'].apply(lambda x: int(x.split('/')[0]))
X_test['season'] = X_test['season'].apply(lambda x: int(x.split('/')[0]))


feature_pipeline.fit(X_train, y_train)

/tmp/ipython-input-1211204085.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['season'] = X_train['season'].apply(lambda x: int(x.split('/')[0]))
/tmp/ipython-input-1211204085.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test['season'] = X_test['season'].apply(lambda x: int(x.split('/')[0]))


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('cat', TargetEncoder(),
                                                  ['venue', 'city', 'season',
                                                   'batting_team',
                                                   'bowling_team',
                                                   'experience_level']),
                                                 ('num', StandardScaler(),
                                                  ['avg_runs_last_5',
                                                   'avg_runs_last_10',
                                                   'avg_runs_at_venue',
                                                   'matches_at_venue',
                                                   'matches_played',
                                                   'career_avg_runs',
                                                   'career_avg_balls',
                                                   'recent_form_indicator'])]))])

In [ ]:
import numpy as np

X_train_transformed = feature_pipeline.transform(X_train)

# Check if X_test is empty before transforming
if not X_test.empty:
    X_test_transformed  = feature_pipeline.transform(X_test)
    print("Train shape:", X_train_transformed.shape)
    print("Test shape:", X_test_transformed.shape)
else:
    # If X_test is empty, create an empty numpy array with the correct number of columns
    # The number of columns should match the transformed X_train.
    X_test_transformed = np.empty((0, X_train_transformed.shape[1]))
    print("Train shape:", X_train_transformed.shape)
    print("Warning: X_test is empty due to the date split condition. No test data transformed.")
    print("Test shape:", X_test_transformed.shape)

Train shape: (9331, 14)
Test shape: (2349, 14)


In [ ]:
joblib.dump(feature_pipeline, "Batsman_Feature_Pipeline.pkl")

print("✅ Batsman feature pipeline saved successfully")


✅ Batsman feature pipeline saved successfully
